<a href="https://colab.research.google.com/github/hruthikcm/TCP-IP-data-Tracing-System/blob/main/CN__PRO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%%writefile app.py
from flask import Flask, render_template, request, jsonify
import random
import hashlib
import time

app = Flask(__name__)

def checksum(text):
    return hashlib.md5(text.encode()).hexdigest()[:8]

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/simulate", methods=["POST"])
def simulate():
    data = request.get_json(silent=True) or {}
    message = data.get("message", "")
    error_type = data.get("error", "none")

    try:
        probability = float(data.get("probability", 30))
    except (TypeError, ValueError):
        return jsonify(success=False, error="Invalid probability"), 400

    if not isinstance(message, str) or not message.strip():
        return jsonify(success=False, error="Please enter a message."), 400

    if len(message) > 2000:
        return jsonify(success=False, error="Maximum message length is 2000 characters."), 400

    if error_type not in {"none", "bit_error", "packet_loss", "delay", "duplicate"}:
        return jsonify(success=False, error="Invalid error type."), 400

    if not 0 <= probability <= 100:
        return jsonify(success=False, error="Probability must be between 0 and 100."), 400

    original = message
    layers = [
        {
            "layer": "Application Layer",
            "protocol": "HTTP / Application Data",
            "data": original,
            "description": "The application generates the original message."
        },
        {
            "layer": "Transport Layer",
            "protocol": "TCP",
            "data": {
                "source_port": 5000,
                "destination_port": 8080,
                "sequence_number": 1001,
                "acknowledgment_number": 0,
                "payload": original
            },
            "description": "TCP adds port and sequence information to a segment."
        },
        {
            "layer": "Internet Layer",
            "protocol": "IPv4",
            "data": {
                "source_ip": "192.168.1.10",
                "destination_ip": "192.168.1.20",
                "protocol": "TCP",
                "ttl": 64,
                "payload": original
            },
            "description": "IP adds logical addressing and routing information."
        },
        {
            "layer": "Network Access Layer",
            "protocol": "Ethernet",
            "data": {
                "source_mac": "AA:BB:CC:DD:EE:01",
                "destination_mac": "AA:BB:CC:DD:EE:02",
                "ether_type": "IPv4",
                "payload": original
            },
            "description": "Ethernet adds local network addressing information."
        }
    ]

    received = original
    error_occurred = False
    description = "No communication error occurred."
    detection = "The received message matches the original message."

    if error_type != "none" and random.random() * 100 < probability:
        error_occurred = True

        if error_type == "bit_error":
            index = random.randrange(len(original))
            old_char = original[index]
            choices = [c for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
                       if c != old_char]
            new_char = random.choice(choices)
            received = original[:index] + new_char + original[index + 1:]
            description = f"Simulated corruption at character position {index}: '{old_char}' changed to '{new_char}'."
            detection = "The message fingerprint differs. Real TCP uses a checksum to detect segment corruption."

        elif error_type == "packet_loss":
            received = ""
            description = "The simulated packet was lost during transmission."
            detection = "Real TCP can detect missing data through sequence numbers and acknowledgments, then retransmit."

        elif error_type == "delay":
            time.sleep(1)
            description = "A one-second transmission delay was simulated."
            detection = "Delay alone does not change data. TCP uses acknowledgments and timers to manage delivery."

        elif error_type == "duplicate":
            received = original + original
            description = "The simulated payload was duplicated."
            detection = "Real TCP uses sequence numbers to reassemble data and handle retransmitted segments."

    original_hash = checksum(original)
    received_hash = checksum(received)
    match = original == received

    return jsonify({
        "success": True,
        "original_message": original,
        "received_message": received,
        "status": "SUCCESS" if match else "ERROR",
        "error_occurred": error_occurred,
        "error_description": description,
        "detection": detection,
        "original_checksum": original_hash,
        "received_checksum": received_hash,
        "layers": layers,
        "statistics": {
            "data_length": len(original),
            "received_length": len(received),
            "checksum_match": original_hash == received_hash,
            "data_match": match
        }
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)

Overwriting app.py


In [11]:
import os
os.makedirs("templates", exist_ok=True)
os.makedirs("static", exist_ok=True)

In [12]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>TCP/IP Data Tracing System</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<header>
  <h1>TCP/IP Data Tracing System</h1>
  <p>Tracing Data Changes Across TCP/IP Layers Under Communication Errors</p>
</header>

<main>
  <section class="card">
    <h2>1. Configure Transmission</h2>

    <label for="message">Message to transmit</label>
    <textarea id="message" maxlength="2000">Hello Computer Networks</textarea>

    <label for="error">Communication error</label>
    <select id="error">
      <option value="none">No Error</option>
      <option value="bit_error">Data Corruption</option>
      <option value="packet_loss">Packet Loss</option>
      <option value="delay">Network Delay</option>
      <option value="duplicate">Packet Duplication</option>
    </select>

    <label for="probability">
      Error probability: <span id="probValue">100%</span>
    </label>

    <input id="probability" type="range" min="0" max="100" value="100">

    <button id="runButton" onclick="simulate()">
      Simulate Transmission
    </button>

    <p id="loading" class="hidden">Processing transmission...</p>
    <p id="errorMessage" class="hidden"></p>
  </section>

  <section id="resultSection" class="card hidden">
    <h2>2. Transmission Result</h2>

    <div id="status" class="status"></div>

    <div class="grid">
      <div>
        <h3>Original Data</h3>
        <pre id="original" class="data"></pre>
      </div>

      <div>
        <h3>Received Data</h3>
        <pre id="received" class="data"></pre>
      </div>
    </div>
  </section>

  <section id="errorSection" class="card hidden">
    <h2>3. Communication Error Analysis</h2>

    <p id="errorDescription"></p>

    <div class="info">
      <strong>TCP/IP handling:</strong>
      <p id="detection"></p>
    </div>
  </section>

  <section id="layersSection" class="card hidden">
    <h2>4. TCP/IP Layer Trace</h2>
    <div id="layers"></div>
  </section>

  <section id="checksumSection" class="card hidden">
    <h2>5. Data Integrity</h2>

    <p class="note">
      These are MD5 fingerprints for comparing the simulated messages,
      not actual TCP checksums.
    </p>

    <div class="grid">
      <div>
        <h3>Original Fingerprint</h3>
        <pre id="originalChecksum" class="data"></pre>
      </div>

      <div>
        <h3>Received Fingerprint</h3>
        <pre id="receivedChecksum" class="data"></pre>
      </div>
    </div>

    <p id="checksumStatus" class="status"></p>
  </section>

  <section id="statisticsSection" class="card hidden">
    <h2>6. Transmission Statistics</h2>

    <div class="grid stats">
      <div class="stat">
        <strong id="dataLength">0</strong>
        <p>Original characters</p>
      </div>

      <div class="stat">
        <strong id="receivedLength">0</strong>
        <p>Received characters</p>
      </div>

      <div class="stat">
        <strong id="integrity">—</strong>
        <p>Data integrity</p>
      </div>
    </div>
  </section>
</main>

<footer>
  Computer Networks Mini Project · TCP/IP Error Simulation
</footer>

<script src="{{ url_for('static', filename='script.js') }}"></script>
</body>
</html>

Overwriting templates/index.html


In [13]:
%%writefile static/style.css
* { box-sizing: border-box; }
body {
  margin: 0; font-family: Arial, sans-serif;
  background: #f1f5f9; color: #1e293b;
}
header {
  background: #172554; color: white; text-align: center;
  padding: 28px 16px;
}
header h1 { margin: 0 0 10px; }
header p { margin: 0; }
main { max-width: 1050px; width: 92%; margin: 28px auto; }
.card {
  background: white; padding: 24px; margin-bottom: 22px;
  border-radius: 12px; box-shadow: 0 3px 12px #0f172a12;
}
h2 { color: #1e3a8a; margin-top: 0; }
label { display: block; font-weight: bold; margin: 16px 0 8px; }
textarea, select {
  width: 100%; padding: 12px; border: 1px solid #cbd5e1;
  border-radius: 7px; font-size: 15px;
}
textarea { min-height: 100px; resize: vertical; }
input[type=range] { width: 100%; }
button {
  width: 100%; padding: 14px; margin-top: 20px; border: 0;
  border-radius: 8px; background: #2563eb; color: white;
  font-size: 16px; font-weight: bold; cursor: pointer;
}
button:hover { background: #1d4ed8; }
button:disabled { opacity: .6; cursor: wait; }
.hidden { display: none; }
.grid { display: grid; grid-template-columns: 1fr 1fr; gap: 18px; }
.data {
  background: #f8fafc; border: 1px solid #cbd5e1;
  padding: 14px; border-radius: 7px; min-height: 55px;
  white-space: pre-wrap; overflow-wrap: anywhere;
}
.status { padding: 13px; border-radius: 7px; font-weight: bold; }
.good { background: #dcfce7; color: #166534; }
.bad { background: #fee2e2; color: #991b1b; }
.info { background: #eff6ff; border-left: 4px solid #2563eb; padding: 14px; margin-top: 18px; }
.layer { border: 1px solid #cbd5e1; border-radius: 8px; margin: 14px 0; overflow: hidden; }
.layer h3 { background: #1e3a8a; color: white; margin: 0; padding: 12px; }
.layer-body { padding: 14px; }
.layer-body pre { white-space: pre-wrap; overflow-wrap: anywhere; }
.protocol { color: #2563eb; font-weight: bold; }
.stats .stat { text-align: center; background: #eff6ff; padding: 20px; border-radius: 8px; }
.stat strong { font-size: 26px; color: #1d4ed8; }
.note { color: #64748b; font-size: 13px; }
#errorMessage { color: #b91c1c; }
footer { background: #172554; color: white; text-align: center; padding: 20px; }
@media(max-width: 650px) { .grid { grid-template-columns: 1fr; } }

Overwriting static/style.css


In [14]:
%%writefile static/script.js
const $ = id => document.getElementById(id);

$("probability").addEventListener("input", () => {
  $("probValue").textContent = $("probability").value + "%";
});

async function simulate() {
  const message = $("message").value;
  if (!message.trim()) {
    $("errorMessage").textContent = "Please enter a message.";
    $("errorMessage").classList.remove("hidden");
    return;
  }

  $("errorMessage").classList.add("hidden");
  $("loading").classList.remove("hidden");
  $("runButton").disabled = true;

  try {
    const response = await fetch("/simulate", {
      method: "POST",
      headers: {"Content-Type": "application/json"},
      body: JSON.stringify({
        message,
        error: $("error").value,
        probability: Number($("probability").value)
      })
    });

    const d = await response.json();
    if (!response.ok || !d.success) throw new Error(d.error || "Request failed");

    ["resultSection", "errorSection", "layersSection",
     "checksumSection", "statisticsSection"].forEach(id =>
      $(id).classList.remove("hidden")
    );

    $("status").textContent = d.status === "SUCCESS"
      ? "✓ DATA RECEIVED — CONTENT MATCHES"
      : "⚠ DATA CHANGED OR LOST";
    $("status").className = "status " + (d.status === "SUCCESS" ? "good" : "bad");

    $("original").textContent = d.original_message;
    $("received").textContent = d.received_message || "NO DATA RECEIVED";
    $("errorDescription").textContent = d.error_description;
    $("detection").textContent = d.detection;

    const layers = $("layers");
    layers.replaceChildren();

    d.layers.forEach(layer => {
      const box = document.createElement("div");
      box.className = "layer";

      const heading = document.createElement("h3");
      heading.textContent = layer.layer + " — " + layer.protocol;

      const body = document.createElement("div");
      body.className = "layer-body";

      const description = document.createElement("p");
      description.textContent = layer.description;

      const pre = document.createElement("pre");
      pre.textContent = JSON.stringify(layer.data, null, 2);

      body.append(description, pre);
      box.append(heading, body);
      layers.appendChild(box);
    });

    $("originalChecksum").textContent = d.original_checksum;
    $("receivedChecksum").textContent = d.received_checksum;

    const matches = d.statistics.checksum_match;
    $("checksumStatus").textContent = matches
      ? "✓ Fingerprints match"
      : "⚠ Fingerprints differ";
    $("checksumStatus").className = "status " + (matches ? "good" : "bad");

    $("dataLength").textContent = d.statistics.data_length;
    $("receivedLength").textContent = d.statistics.received_length;
    $("integrity").textContent = d.statistics.data_match ? "INTACT" : "CHANGED";

    $("resultSection").scrollIntoView({behavior: "smooth"});
  } catch (e) {
    $("errorMessage").textContent = "Error: " + e.message;
    $("errorMessage").classList.remove("hidden");
  } finally {
    $("loading").classList.add("hidden");
    $("runButton").disabled = false;
  }
}

Overwriting static/script.js


In [15]:
import subprocess
import time
import requests

# Stop an earlier server if this cell is run again
if "flask_process" in globals() and flask_process.poll() is None:
    flask_process.terminate()
    flask_process.wait()

flask_process = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

for i in range(20):
    try:
        r = requests.get("http://127.0.0.1:5000/", timeout=2)
        if r.status_code == 200:
            print("Flask server started successfully!")
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Flask did not start. Check app.py for errors.")

Flask server started successfully!


In [16]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(5000)")
print("Open your project here:")
print(url)

Open your project here:
https://5000-m-s-kkb-usw3b2-27p2fdu2lflnj-b.us-west3-2.prod.colab.dev
